# 模組 4：用遺傳演算法（GA）挑投資組合 ＋ 揪過度最佳化

用現成的 GA 套件 `PyGAD`，即時抓真實的 0050 成分股股價，讓它自動找一組「權重」湊出夏普比（划算程度）最高的投資組合；再切樣本內／外，看漂亮的回測拿到「沒看過的未來」還剩幾分——這就是**過度最佳化**。

> 需要網路（要即時抓股價）。不需 API / Ollama，純本地算。

> ## ⚠️ 今天的主角：過度最佳化（先讀）
> 今天用的是**真實的 0050 成分股**，整堂課只示範「怎麼做最佳化、怎麼揪過度最佳化」。GA 挑出來的投組漂不漂亮，看的是回測數字，**不代表未來會賺**。最終決策要由人判斷（human-in-the-loop）。今天最該記住的是「**別信漂亮的回測**」。

## 🟦 A 段：今天要做什麼

一句話：**找一組權重，讓一籃子股票的「夏普比」最高**。

- **投資組合**＝同時買好幾檔、各給一個**權重**（各買多少比例）。
- **夏普比**＝這籃子「賺得穩不穩」的分數，越高越划算。
- **GA（遺傳演算法）** ＝模仿生物演化，一代一代自動試出好權重的搜尋法；內部四步（算分／選擇／交配／突變）點到即止，細節看投影片與課堂講解。
- **PyGAD**＝現成的 GA 輪子，我們不用自己刻，只要把「要最大化的分數」餵給它。

> 概念與生活類比在課堂講解，這裡直接動手。

## 🟩 B 段：抓資料 → 算報酬 → 讓 PyGAD 找最佳投組

先裝套件。已裝好的會直接跳過；裝不起來看 README。

In [1]:
# 【格1】裝套件
!pip install -q pygad==3.7.0 yfinance==1.3.0    # 💡 -q＝安靜安裝，少洗版；pygad＝現成 GA 輪子、yfinance＝抓股價


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
# import 今天要用的套件
import numpy as np, pandas as pd, yfinance as yf, pygad

### 先抓一檔看看：yfinance 怎麼用

抓一整籃之前，先拿**台積電（代號 `2330.TW`）** 一檔試手，看 `yf.download()` 回什麼。台股代號後面要加 `.TW`。

In [3]:
# 先抓「台積電 2330.TW」一檔，熟悉 yf.download 的用法
stock_2330 = yf.download("2330.TW", period="4y", interval="1d", auto_adjust=True, progress=False)
stock_2330.tail()

Price,Close,High,Low,Open,Volume
Ticker,2330.TW,2330.TW,2330.TW,2330.TW,2330.TW
Date,,,,,
2026-07-15,2440.0,2460.0,2415.0,2425.0,28532735
2026-07-16,2470.0,2470.0,2420.0,2430.0,27848158
2026-07-17,2290.0,2395.0,2290.0,2375.0,75253147
2026-07-20,2320.0,2345.0,2300.0,2300.0,45272691
2026-07-21,NaN,NaN,NaN,NaN,31271386


### 0050 成分股清單

熟悉單檔後，換成一整籃 0050 成分股的代號。

> 🎯 這是大型權值股清單；**正式成分股以當期公告為準**，清單會換。抓不到資料的（下市、改代號）後面會自動剔除，不用手動改。

In [4]:
# 0050 成分股（大型權值股；正式清單以當期公告為準，抓不到的自動剔除）
tickers = ["2330","2317","2454","2308","2382","2881","2882","2412","2891","2303",
           "3711","2886","2884","1301","1303","2002","2207","3008","2357","2379",
           "3034","2395","2345","2890","2892","5880","2885","1216","2603","2609",
           "2615","3037","3231","2356","4938","6505","1101","2409","3045","2327",
           "2408","1326","2474","6415","3661"]
tw = [t + ".TW" for t in tickers]    # 💡 yfinance 要「代號.TW」才認得台股（上市）

### 即時抓近 4 年股價

跟 yfinance 要一整籃近 4 年的每日收盤價，然後**分兩步清乾淨**：先丟「整欄都沒資料」的股票，再丟「還有缺值」的天。

> 🎲 **即時抓，每次跑到的資料都略不同**——後面所有數字看趨勢、不對精確值。
> 🌐 抓股價要連網；跑很慢或抓到 0 檔多半是網路擋 yfinance，重跑或看 README。

In [5]:
# 【格3】即時抓近 4 年股價
raw = yf.download(tw, period="4y", interval="1d", auto_adjust=True, progress=False)  # 💡 auto_adjust＝還原除權息，報酬才不會失真
close = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw    # 💡 只抓到 1 檔時欄位長相不同，這行兩種都接得住
close    # 先看一眼原始長相（有些欄可能整欄 NaN、有些天有缺值）

Ticker,1101.TW,1216.TW,1301.TW,1303.TW,1326.TW,2002.TW,2207.TW,2303.TW,2308.TW,2317.TW,...,3034.TW,3037.TW,3045.TW,3231.TW,3661.TW,3711.TW,4938.TW,5880.TW,6415.TW,6505.TW
Date,,,,,,,,,,,,,,,,,,,,,
2022-07-21,33.877480,58.104496,81.398865,59.809372,62.902767,25.710339,516.742126,34.914509,233.135193,91.945084,...,211.673645,164.038422,84.023651,22.293285,665.817322,73.613419,50.057148,20.226362,605.254578,76.286972
2022-07-22,33.565861,59.559250,80.573860,59.533325,62.995686,25.963226,517.619446,34.628994,231.270096,92.380844,...,208.924637,163.576355,84.023651,22.293285,653.291138,73.183937,50.388107,20.265261,597.482483,75.645905
2022-07-25,33.699409,59.901543,82.773834,60.361454,64.017731,26.384708,524.638000,33.038265,233.601440,93.688110,...,205.782913,162.652191,85.210670,22.382105,652.327637,72.668556,50.967278,20.615334,561.536316,76.836456
2022-07-26,33.654892,60.329414,82.773834,60.453472,63.553165,26.057238,526.392639,31.488319,232.202621,94.123878,...,205.390198,157.107224,85.210670,22.559738,640.764954,71.895493,50.305367,20.615334,545.992126,76.470131
2022-07-27,33.699409,60.243835,82.682167,60.637501,63.831902,25.823332,525.515320,32.181717,236.865341,94.123878,...,207.746490,165.886749,86.482468,23.092644,656.181824,73.098038,50.967278,20.965403,547.935120,76.928032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-15,23.549999,78.599998,65.699997,227.500000,70.800003,18.650000,480.000000,166.000000,1890.000000,239.000000,...,472.000000,936.000000,109.000000,145.500000,3735.000000,683.000000,82.699997,25.350000,504.000000,72.800003
2026-07-16,23.950001,79.000000,65.599998,221.000000,68.900002,18.799999,493.000000,160.000000,1905.000000,242.500000,...,471.500000,882.000000,110.500000,146.500000,3770.000000,682.000000,83.199997,25.500000,473.000000,80.000000
2026-07-17,23.500000,80.000000,62.799999,199.000000,66.099998,18.650000,486.000000,144.000000,1740.000000,234.000000,...,460.500000,794.000000,111.000000,139.000000,3480.000000,614.000000,82.199997,25.350000,435.000000,81.300003


In [6]:
# 第 1 步：丟掉「整欄都是空值」的股票（近 4 年完全沒資料的）
step_1 = close.dropna(axis=1, how="all")    # 💡 axis=1＝看「欄」，how="all"＝整欄全空才丟
step_1

Ticker,1101.TW,1216.TW,1301.TW,1303.TW,1326.TW,2002.TW,2207.TW,2303.TW,2308.TW,2317.TW,...,3034.TW,3037.TW,3045.TW,3231.TW,3661.TW,3711.TW,4938.TW,5880.TW,6415.TW,6505.TW
Date,,,,,,,,,,,,,,,,,,,,,
2022-07-21,33.877480,58.104496,81.398865,59.809372,62.902767,25.710339,516.742126,34.914509,233.135193,91.945084,...,211.673645,164.038422,84.023651,22.293285,665.817322,73.613419,50.057148,20.226362,605.254578,76.286972
2022-07-22,33.565861,59.559250,80.573860,59.533325,62.995686,25.963226,517.619446,34.628994,231.270096,92.380844,...,208.924637,163.576355,84.023651,22.293285,653.291138,73.183937,50.388107,20.265261,597.482483,75.645905
2022-07-25,33.699409,59.901543,82.773834,60.361454,64.017731,26.384708,524.638000,33.038265,233.601440,93.688110,...,205.782913,162.652191,85.210670,22.382105,652.327637,72.668556,50.967278,20.615334,561.536316,76.836456
2022-07-26,33.654892,60.329414,82.773834,60.453472,63.553165,26.057238,526.392639,31.488319,232.202621,94.123878,...,205.390198,157.107224,85.210670,22.559738,640.764954,71.895493,50.305367,20.615334,545.992126,76.470131
2022-07-27,33.699409,60.243835,82.682167,60.637501,63.831902,25.823332,525.515320,32.181717,236.865341,94.123878,...,207.746490,165.886749,86.482468,23.092644,656.181824,73.098038,50.967278,20.965403,547.935120,76.928032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-15,23.549999,78.599998,65.699997,227.500000,70.800003,18.650000,480.000000,166.000000,1890.000000,239.000000,...,472.000000,936.000000,109.000000,145.500000,3735.000000,683.000000,82.699997,25.350000,504.000000,72.800003
2026-07-16,23.950001,79.000000,65.599998,221.000000,68.900002,18.799999,493.000000,160.000000,1905.000000,242.500000,...,471.500000,882.000000,110.500000,146.500000,3770.000000,682.000000,83.199997,25.500000,473.000000,80.000000
2026-07-17,23.500000,80.000000,62.799999,199.000000,66.099998,18.650000,486.000000,144.000000,1740.000000,234.000000,...,460.500000,794.000000,111.000000,139.000000,3480.000000,614.000000,82.199997,25.350000,435.000000,81.300003


In [7]:
# 第 2 步：再丟掉「還有缺值的那幾天」→ 只留全程都有資料的乾淨表
step_2 = step_1.dropna()    # 💡 不給參數＝預設丟「含任何缺值的列（天）」
step_2

Ticker,1101.TW,1216.TW,1301.TW,1303.TW,1326.TW,2002.TW,2207.TW,2303.TW,2308.TW,2317.TW,...,3034.TW,3037.TW,3045.TW,3231.TW,3661.TW,3711.TW,4938.TW,5880.TW,6415.TW,6505.TW
Date,,,,,,,,,,,,,,,,,,,,,
2022-07-21,33.877480,58.104496,81.398865,59.809372,62.902767,25.710339,516.742126,34.914509,233.135193,91.945084,...,211.673645,164.038422,84.023651,22.293285,665.817322,73.613419,50.057148,20.226362,605.254578,76.286972
2022-07-22,33.565861,59.559250,80.573860,59.533325,62.995686,25.963226,517.619446,34.628994,231.270096,92.380844,...,208.924637,163.576355,84.023651,22.293285,653.291138,73.183937,50.388107,20.265261,597.482483,75.645905
2022-07-25,33.699409,59.901543,82.773834,60.361454,64.017731,26.384708,524.638000,33.038265,233.601440,93.688110,...,205.782913,162.652191,85.210670,22.382105,652.327637,72.668556,50.967278,20.615334,561.536316,76.836456
2022-07-26,33.654892,60.329414,82.773834,60.453472,63.553165,26.057238,526.392639,31.488319,232.202621,94.123878,...,205.390198,157.107224,85.210670,22.559738,640.764954,71.895493,50.305367,20.615334,545.992126,76.470131
2022-07-27,33.699409,60.243835,82.682167,60.637501,63.831902,25.823332,525.515320,32.181717,236.865341,94.123878,...,207.746490,165.886749,86.482468,23.092644,656.181824,73.098038,50.967278,20.965403,547.935120,76.928032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-14,22.700001,78.500000,63.599998,210.000000,67.000000,18.500000,480.000000,151.000000,1855.000000,235.500000,...,468.000000,874.000000,109.000000,142.500000,3725.000000,641.000000,82.099998,25.200001,493.500000,66.199997
2026-07-15,23.549999,78.599998,65.699997,227.500000,70.800003,18.650000,480.000000,166.000000,1890.000000,239.000000,...,472.000000,936.000000,109.000000,145.500000,3735.000000,683.000000,82.699997,25.350000,504.000000,72.800003
2026-07-16,23.950001,79.000000,65.599998,221.000000,68.900002,18.799999,493.000000,160.000000,1905.000000,242.500000,...,471.500000,882.000000,110.500000,146.500000,3770.000000,682.000000,83.199997,25.500000,473.000000,80.000000


In [8]:
close = step_2
print("抓到", close.shape[1], "檔、", len(close), "個交易日")
close.tail()

抓到 45 檔、 968 個交易日


Ticker,1101.TW,1216.TW,1301.TW,1303.TW,1326.TW,2002.TW,2207.TW,2303.TW,2308.TW,2317.TW,...,3034.TW,3037.TW,3045.TW,3231.TW,3661.TW,3711.TW,4938.TW,5880.TW,6415.TW,6505.TW
Date,,,,,,,,,,,,,,,,,,,,,
2026-07-14,22.700001,78.500000,63.599998,210.0,67.000000,18.500000,480.0,151.0,1855.0,235.5,...,468.0,874.0,109.0,142.5,3725.0,641.0,82.099998,25.200001,493.5,66.199997
2026-07-15,23.549999,78.599998,65.699997,227.5,70.800003,18.650000,480.0,166.0,1890.0,239.0,...,472.0,936.0,109.0,145.5,3735.0,683.0,82.699997,25.350000,504.0,72.800003
2026-07-16,23.950001,79.000000,65.599998,221.0,68.900002,18.799999,493.0,160.0,1905.0,242.5,...,471.5,882.0,110.5,146.5,3770.0,682.0,83.199997,25.500000,473.0,80.000000
2026-07-17,23.500000,80.000000,62.799999,199.0,66.099998,18.650000,486.0,144.0,1740.0,234.0,...,460.5,794.0,111.0,139.0,3480.0,614.0,82.199997,25.350000,435.0,81.300003
2026-07-20,23.799999,80.300003,60.599998,186.5,64.199997,18.549999,490.0,130.0,1705.0,234.5,...,458.0,750.0,112.0,143.5,3365.0,597.0,80.000000,25.500000,427.5,77.699997


### 算每日報酬

把股價轉成「每天漲跌幾 %」——這就是等一下算夏普比、給 GA 最大化的原料。

In [9]:
# 算每日報酬
rets = close.pct_change().dropna()    # 💡 相對前一天的變化比例；第一天沒昨天可比，dropna 丟掉
rets

Ticker,1101.TW,1216.TW,1301.TW,1303.TW,1326.TW,2002.TW,2207.TW,2303.TW,2308.TW,2317.TW,...,3034.TW,3037.TW,3045.TW,3231.TW,3661.TW,3711.TW,4938.TW,5880.TW,6415.TW,6505.TW
Date,,,,,,,,,,,,,,,,,,,,,
2022-07-22,-0.009198,0.025037,-0.010135,-0.004615,0.001477,0.009836,0.001698,-0.008178,-0.008000,0.004739,...,-0.012987,-0.002817,0.000000,0.000000,-0.018813,-0.005834,0.006612,0.001923,-0.012841,-0.008403
2022-07-25,0.003979,0.005747,0.027304,0.013910,0.016224,0.016234,0.013559,-0.045936,0.010081,0.014151,...,-0.015038,-0.005650,0.014127,0.003984,-0.001475,-0.007042,0.011494,0.017275,-0.060163,0.015738
2022-07-26,-0.001321,0.007143,0.000000,0.001524,-0.007257,-0.012411,0.003344,-0.046914,-0.005988,0.004651,...,-0.001908,-0.034091,0.000000,0.007936,-0.017725,-0.010638,-0.012987,0.000000,-0.027682,-0.004768
2022-07-27,0.001323,-0.001419,-0.001107,0.003044,0.004386,-0.008977,-0.001667,0.022021,0.020080,0.000000,...,0.011472,0.055882,0.014925,0.023622,0.024060,0.016726,0.013158,0.016981,0.003559,0.005988
2022-07-28,0.009247,-0.007102,0.009978,0.010622,0.001456,-0.009058,0.006678,-0.016477,0.031496,0.004630,...,-0.007561,-0.097493,-0.004902,0.013462,-0.007342,0.031727,0.011364,0.011132,0.000000,0.001191
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-14,-0.006565,0.000000,0.056478,0.052632,0.030769,0.005435,0.010526,-0.016287,-0.018519,-0.004228,...,0.001070,-0.032115,-0.004566,-0.006969,-0.099154,-0.043284,-0.006053,0.000000,-0.058206,0.049128
2026-07-15,0.037445,0.001274,0.033019,0.083333,0.056716,0.008108,0.000000,0.099338,0.018868,0.014862,...,0.008547,0.070938,0.000000,0.021053,0.002685,0.065523,0.007308,0.005952,0.021277,0.099698
2026-07-16,0.016985,0.005089,-0.001522,-0.028571,-0.026836,0.008043,0.027083,-0.036145,0.007937,0.014644,...,-0.001059,-0.057692,0.013761,0.006873,0.009371,-0.001464,0.006046,0.005917,-0.061508,0.098901


In [10]:
N = rets.shape[1]
print("資產數：", N)

資產數： 45


### 讓 PyGAD 找最佳投組（先用最近 60 天當範例）

先拿最近 60 天（約一季／季線）練手，建立 PyGAD 的用法，看它挑出的投組夏普比多高。

> 🎯 **60 天＝季線**，技術分析常用的窗。
> 🕹️ 我們只餵「要最大化的分數」（夏普比）給 PyGAD，**GA 內部怎麼演化是黑盒子**——原理看投影片，這裡只看結果。

In [11]:
WIN = 60
recent = rets.tail(WIN)    # 💡 取最後 60 列＝最近 60 個交易日
recent

Ticker,1101.TW,1216.TW,1301.TW,1303.TW,1326.TW,2002.TW,2207.TW,2303.TW,2308.TW,2317.TW,...,3034.TW,3037.TW,3045.TW,3231.TW,3661.TW,3711.TW,4938.TW,5880.TW,6415.TW,6505.TW
Date,,,,,,,,,,,,,,,,,,,,,
2026-04-24,-0.004073,-0.005563,-0.015686,0.008216,-0.001012,-0.007833,-0.018887,0.012228,0.047980,-0.015556,...,0.029557,0.085165,0.008929,0.014337,0.059045,0.067815,0.003572,-0.006479,0.049338,-0.001934
2026-04-27,-0.004090,-0.013986,0.001992,0.022119,0.021277,-0.013158,-0.013171,-0.024161,-0.026506,0.029345,...,-0.003589,0.062025,-0.008850,0.007067,-0.013049,-0.001008,-0.014235,-0.004348,-0.036697,0.003876
2026-04-28,0.002053,0.001418,0.015905,0.019362,0.019841,0.000000,-0.005133,0.033012,0.051980,-0.010965,...,-0.009604,-0.016687,-0.008929,-0.014035,-0.038462,0.000000,-0.004814,0.002183,-0.016667,0.021236
2026-04-29,0.004098,-0.008499,-0.001957,0.023464,0.021401,0.013333,-0.009288,-0.007989,0.018824,-0.002217,...,-0.009697,-0.026667,0.009009,0.000000,0.001250,-0.014127,-0.008464,0.008715,-0.048426,0.020794
2026-04-30,0.000000,-0.010000,0.009804,-0.021834,-0.003810,-0.013158,-0.007292,0.037584,0.000000,-0.024444,...,0.001224,0.099626,-0.004464,-0.024911,0.032459,-0.021494,0.003659,-0.008639,0.086514,0.044444
2026-05-04,-0.002041,0.004329,-0.013592,-0.013393,-0.028681,-0.008000,-0.007345,0.050453,0.025404,0.036446,...,0.015892,0.031710,-0.008969,0.025547,0.027811,0.098326,-0.012151,-0.002179,0.018735,-0.042553
2026-05-05,-0.014315,-0.007184,0.009843,0.063348,0.019685,0.000000,0.010571,0.023399,-0.024775,0.052747,...,0.010830,-0.008782,0.000000,0.010676,-0.023529,-0.009524,0.002460,0.000000,0.002299,0.025926
2026-05-06,0.039419,-0.005789,-0.011696,-0.002128,-0.015444,0.010753,-0.008368,0.099880,0.020785,0.052192,...,0.051190,-0.049834,-0.004525,0.031690,0.100000,0.007692,0.025767,0.002183,-0.027523,-0.010830
2026-05-07,0.017964,0.010189,-0.022683,-0.037313,-0.030392,0.002660,0.027426,0.055799,0.031674,0.005952,...,0.026048,0.044289,0.000000,-0.003413,0.050383,0.030534,0.008373,0.002179,0.001179,-0.045620


In [12]:
def sharpe(weights, R):
    """夏普比（年化）= 投組平均報酬 / 報酬標準差 × √252。越高越划算。"""
    w = np.clip(np.asarray(weights, float), 0, None)   # 💡 負權重壓成 0＝不放空、只做多
    if w.sum() == 0:
        return 0.0
    w = w / w.sum()                                    # 💡 正規化成合法權重（加總＝1），GA 亂丟的數字也能變合法比例
    port = R.values @ w                                # 💡 每一天的投組報酬＝各股報酬 × 各自權重再加總
    return 0.0 if port.std() == 0 else port.mean() / port.std() * np.sqrt(252)  # 💡 ×√252＝把「每日」放大成「年化」

def fitness_func(ga_instance, solution, solution_idx):
    return sharpe(solution, recent)          # 💡 PyGAD 要最大化的目標＝夏普比；solution＝它試的一組權重

ga = pygad.GA(num_generations=120,           # 💡 演化 120 代
              num_parents_mating=12,          # 每代選 12 個當父母
              fitness_func=fitness_func,
              sol_per_pop=40,                 # 族群 40 個候選投組
              num_genes=N,                    # 💡 每個候選＝N 檔的權重，所以基因數＝資產數
              gene_space={'low': 0.0, 'high': 1.0},   # 💡 權重介於 0~1
              gene_type=float,
              random_seed=83,                 # 💡 🎲 固定，可重現
              mutation_percent_genes=20,
              suppress_warnings=True)
ga.run()

### GA 跑完了，看它找到什麼

`ga.best_solution()` 回傳三樣：**最佳解、最佳分數、索引**。先把「最佳解」印出來看看它長什麼樣子。

In [13]:
best, best_fit, _ = ga.best_solution()
best    # 💡 這是 GA 找到的一組「原始權重」——還沒正規化，可能有負、加起來也不是 1

array([0.49910826, 0.55278994, 0.26162013, 0.96331384, 0.22597524,
       0.79495163, 0.13625922, 0.57385046, 0.05736769, 0.27628668,
       0.93308761, 0.02747164, 0.03283278, 0.04496994, 0.4131938 ,
       0.36913172, 0.10625774, 0.5665295 , 0.24011582, 0.34777103,
       0.20541328, 0.08792876, 0.05486354, 0.40361366, 0.10893709,
       0.45262026, 0.82303383, 0.95755913, 0.24529285, 0.87860147,
       0.93826999, 0.092072  , 0.79057091, 0.49977713, 0.89544772,
       0.31370037, 0.01299944, 0.3793786 , 0.04586627, 0.03931822,
       0.29404253, 0.05785855, 0.55287693, 0.05172951, 0.79739535])

In [14]:
best_w = np.clip(best, 0, None)
best_w = best_w / best_w.sum()   # 💡 把原始權重壓非負＋正規化成合法投組（加總＝1）
print("GA 找到的最佳投組夏普比：", round(sharpe(best_w, recent), 2))

GA 找到的最佳投組夏普比： 5.27


In [15]:
# 看它押最重的前 5 檔
top5 = np.argsort(best_w)[::-1][:5]    # 💡 argsort 由小到大排索引，[::-1] 反轉成由大到小，取前 5
for i in top5:
    print(f"  {rets.columns[i]}  權重 {round(best_w[i]*100, 1)}%")

  1303.TW  權重 5.5%
  2882.TW  權重 5.5%
  2886.TW  權重 5.4%
  2327.TW  權重 5.4%
  3008.TW  權重 5.1%


> 說明：這 5 檔只是「這 60 天回測最划算」的組合，**不是推薦買它們**。下一段就會看到，換一段時間考它，這組合可能就不靈了。

## 🟧 C 段：切樣本內／外 — 揪過度最佳化

前面 GA 在同一段資料裡「又找答案又打分」，當然漂亮。真正的考驗是：**用一段資料找出的投組，拿到另一段沒看過的時間，還剩幾分？**

- **樣本內**＝GA 拿來找權重的那段（念書範圍）。
- **樣本外**＝另一段沒看過的時間（考試）。
- **過度最佳化**＝把回測那段的雜訊也當規律硬記，換段時間就縮水。

### 先複習：用「序號」切一段資料

等一下要用序號把資料切成「前一段／後一段」。先用一個小 list 熱身，看序號切片 `[起:迄]` 怎麼運作（含頭不含尾）。

In [16]:
list_a = [10, 11, 12, 13, 14, 15, 16, 17, 18]
print("list_a[1:3] =", list_a[1:3])   # 從第 1 個到第 3 個「之前」（含頭不含尾）→ 11, 12
print("list_a[3:]  =", list_a[3:])    # 從第 3 個到最後 → 13,14,15,16,17,18
print("倒數 3 個 list_a[-3:] =", list_a[-3:])

list_a[1:3] = [11, 12]
list_a[3:]  = [13, 14, 15, 16, 17, 18]
倒數 3 個 list_a[-3:] = [16, 17, 18]


### 按時間切成「更早 60 天（樣本內）」和「最近 60 天（樣本外）」

`split` ＝總天數減 60，當作分界。樣本內＝`split` 往前 60 天，樣本外＝`split` 到最後。**注意是「時序切」**：較早的念書、較晚的考試，不可隨機打散（那會偷看未來）。

In [17]:
# 【格6】切樣本內 / 樣本外（都是 60 天）
split = len(rets) - WIN
rets_in  = rets.iloc[split - WIN:split]    # 💡 樣本內：更早的 60 天（GA 在這段找權重＝念書）
rets_out = rets.iloc[split:]               # 💡 樣本外：最近的 60 天（拿來考，GA 沒看過）
print(f"樣本內 {len(rets_in)} 天 → 樣本外 {len(rets_out)} 天")

樣本內 60 天 → 樣本外 60 天


### 主示範：GA 只在樣本內找 → 樣本外考 → 對比「平均分配」

規矩改對：GA **只准看樣本內**找權重，再拿樣本外考。同時擺一個「平均分配」（不動腦、每檔一樣多）當基準。

> 👀 **看兩件事：** ①GA 的樣本內 vs 樣本外差多少（縮水＝過度最佳化）；②樣本外時，GA 有沒有贏過「平均分配」。
> ⏳ 這格要跑一次完整 GA，會等幾秒。

In [18]:
# 【格7】把「用一段資料找最佳權重」包成函式（樣本內、掃窗都會用到）
def ga_best_weights(train):    # 💡 餵一段資料 train，回傳「只看這段找到的」最佳權重
    def fit(ga_i, sol, idx): return sharpe(sol, train)   # 💡 只用 train 打分，樣本外完全沒參與訓練
    g = pygad.GA(num_generations=120, num_parents_mating=12, fitness_func=fit,
                 sol_per_pop=40, num_genes=N, gene_space={'low':0.0,'high':1.0},
                 gene_type=float, random_seed=83, mutation_percent_genes=20,
                 suppress_warnings=True)
    g.run()
    w = np.clip(g.best_solution()[0], 0, None); return w / w.sum()

In [19]:
w_ga = ga_best_weights(rets_in)   # 💡 GA 只吃樣本內
eq   = np.ones(N) / N          # 💡 平均分配（不動腦，每檔一樣多）＝最笨的對照組
print(f"{'策略'.ljust(12)}{'樣本內'.rjust(10)}{'樣本外'.rjust(10)}")
print(f"{'GA 最佳'.ljust(12)}{str(round(sharpe(w_ga, rets_in), 2)).rjust(10)}{str(round(sharpe(w_ga, rets_out), 2)).rjust(10)}")   # 💡 同一組 w_ga，左邊念書分、右邊考試分
print(f"{'平均分配'.ljust(12)}{str(round(sharpe(eq, rets_in), 2)).rjust(10)}{str(round(sharpe(eq, rets_out), 2)).rjust(10)}")

策略                 樣本內       樣本外
GA 最佳             4.65      2.51
平均分配               2.8      3.44


### 怎麼讀這張表（即時資料，你那次可能不同）

**保證會看到的主軸：** GA 的**樣本內分數漂亮 → 樣本外縮水**。GA 把樣本內的雜訊也硬背了，換段時間就打回原形，這就是過度最佳化。

**很常見的彩蛋：** GA 的**樣本外還輸給「平均分配」**——辛苦最佳化半天，不如每檔買一樣多。

**萬一你這次樣本外沒縮水：** 代表這段真實市場訊號夠強、GA 這次沒過度最佳化。但——你敢賭下次也這樣嗎？這正是為什麼**永遠要看樣本外**。下面的掃窗作業（尤其 1 個月的短窗）幾乎一定會讓你看到崩給你看。

> 🔗 同一個道理：課程 1 的 Precision／Recall、模組 3 的 k 與樹深、今天的過度最佳化——都是「別只看訓練分數」的同一把誠實的尺。

## 🟫 D 段：收尾 — 一把貫穿全課的誠實的尺

今天把「數據路」三招收官：模組 3 的 KNN 與決策樹、模組 4 GA 最佳化。串起來就是一句：**任何漂亮的模型／回測，都要拿「沒看過的資料」再驗一次。**

```mermaid
flowchart LR
    A[課程1<br/>Precision/Recall] --> B[模組3<br/>選 k]
    B --> C[模組3<br/>樹深]
    C --> D[模組4<br/>GA 過度最佳化]
```

> 🔗 同一把尺，量到底：訓練分數再漂亮，樣本外／實盤才算數。

> ## 🚫 收尾免責（再一次，加重）
> 今天用真實 0050 只為了示範最佳化與過度最佳化。GA 挑的投組在回測漂亮、樣本外就縮水，本身就是「別信漂亮回測」最好的反例。真上場前，一切要人來判斷（human-in-the-loop），**沒有一組權重是「照抄就會賺」**。
>
> 🎓 **iPAS / AI-901 接點：** 最佳化問題、過度最佳化 vs 過擬合、樣本內外驗證、模型評估、human-in-the-loop 的必要性。

## 📝 回家作業：掃不同時間窗，看誰崩最兇

上面用 60 天。作業把窗換成 1 個月 / 3 個月 / 6 個月 / 1 年，各跑一次 GA，比樣本內 → 樣本外崩多少，也對比平均分配。

> 🎯 **要驗證的猜想：窗越短，越容易過度最佳化**（短窗雜訊多、GA 更容易硬背）。
> 下面是完整版參考解，先自己想「哪個窗崩最兇」再看跑出來的結果。

In [20]:
# 【格8・回家作業】掃時間窗：窗越短越容易過度最佳化
for name, W in {"1m":21, "3m":63, "6m":126, "1y":252}.items():   # 💡 交易日：1 月≈21、3 月≈63、6 月≈126、1 年≈252
    train = rets.iloc[split - W:split]     # 💡 樣本內換成 W 天；樣本外都固定用同一段 rets_out 才公平
    w = ga_best_weights(train)
    gi, go, eo = sharpe(w, train), sharpe(w, rets_out), sharpe(eq, rets_out)
    print(f"{name.rjust(4)} 窗：樣本內 {str(round(gi, 2)).rjust(5)} → 樣本外 {str(round(go, 2)).rjust(5)}（崩 {round(go - gi, 2)}）  平均分配樣本外 {str(round(eo, 2)).rjust(5)}")

  1m 窗：樣本內  9.67 → 樣本外  2.97（崩 -6.7）  平均分配樣本外  3.44


  3m 窗：樣本內  4.55 → 樣本外  2.64（崩 -1.91）  平均分配樣本外  3.44


  6m 窗：樣本內  4.07 → 樣本外  3.31（崩 -0.76）  平均分配樣本外  3.44


  1y 窗：樣本內  4.35 → 樣本外  2.92（崩 -1.43）  平均分配樣本外  3.44


### 📌 參考解結論

即時資料每次不同，但趨勢通常是：

- **1 個月窗崩最兇**——短窗雜訊多，GA 把雜訊當規律硬背，樣本外幾乎一定大縮水，還常輸平均分配。
- **窗拉長（6 個月、1 年）縮水通常收斂**——資料多，GA 比較沒空間過度最佳化。

**一句帶走：窗越短、越容易過度最佳化。** 這也是為什麼看回測不能只挑一段漂亮的時間——換段時間、拉長時間再驗，才知道是不是真本事。

## 🛟 附錄：套件驗證（不需網路）

某格連不上 yfinance 時，先跑這個確認 **PyGAD 本身會動**（玩具問題：找 6 個數字讓總和最大）。這幾行能跑，套件層就沒問題，剩下就是網路能不能連 yfinance。

In [99]:
!pip install -q pygad==3.7.0
import pygad
def fit(ga, sol, idx): return sum(sol)                 # fitness = 總和（越大越好）
g = pygad.GA(num_generations=30, num_parents_mating=4, fitness_func=fit,
             sol_per_pop=10, num_genes=6, gene_space={'low':0.0,'high':1.0},
             gene_type=float, random_seed=83, suppress_warnings=True)
g.run()
print("最佳解:", g.best_solution()[0].round(2), "總和:", round(g.best_solution()[1], 2))

最佳解: [0.95 0.98 1.   0.92 0.89 1.  ] 總和: 5.73
